# Projeto #4 - Planejamento de Capacidade na Nuvem

## Equipe

- Carlos Duarte - matr. 2527530
- Jonas de A. Luz Jr. - matr. 2519171

----

In [1]:
import os, re

import pandas as pd

## Objetivo
>
> Fonte: [Especificação do projeto 4](https://docs.google.com/document/d/13QL64Om-XBFfEqDDyZp8czq-vdWQG-e4mvE3uwRpi-c/edit?tab=t.0)

**Da expecificação original:**

- Arquitetar a infraestrutura de um serviço de blog (WordPress) na AWS.
- Desafio: Maximizar o RPS (Requests Per Second) suportado pelo serviço, sujeito às seguintes restrições:
  - Orçamento: O custo da sua camada de aplicação não pode exceder US$0.50/hora (preço On-Demand us-east-1).
  - Qualidade (SLO): Taxa de Erro < 1% e Latência P95 < 10000ms.
  - Componentes Fixos: O Banco de Dados e o Load Balancer são fornecidos pela "Arena" e não podem ser modificados.
- O trabalho é ajustar a camada de aplicação, escolhendo a melhor combinação de escalabilidade vertical (tamanho da máquina) e horizontal (quantidade de máquinas) para implantar o WordPress.

São fornecidos os *scripts base* para a implementação do WordPress, que podem ser encontrados na pasta `scripts` do projeto.

## Estratégia de Implementação

Nossa estratégia de implementação do trabalho foi a seguinte:

1. Converter os scripts originais para o formato de *batch* do PowerShell, uma vez que o trabalho foi realizado em ambiente de desenvolvimento Windows.
2. Levantar os custos operacionais das instâncias de teste, uma vez que o orçamento foi limitado a US$0.50/hora. Os custos oficiais da AWS foram consultados [no site oficial do serviço EC2](https://us-east-1.console.aws.amazon.com/ec2/home?region=us-east-1).
3. Definição de experimentos para teste de escalabilidade vertical e horizontal, com o objetivo de encontrar a melhor combinação de tamanho da máquina e quantidade de máquinas para implantar o WordPress.
4. Implementação de scripts de apoio com o objetivo de automatizar a execução dos experimentos e coleta de métricas.
5. Implementação de scripts de apoio para coleta de métricas e análise de resultados.
6. Realização de modificações na configuração da aplicação WordPress para otimização do desempenho.
7. Avaliação dos resultados dos experimentos e coleta de métricas.
8. Elaboração de relatório com os resultados dos experimentos e coleta de métricas e escolha da melhor combinação de tamanho da máquina e quantidade de máquinas para implantar o WordPress.

O detalhamento das etapas é descrito nas seções seguintes.

## Implementação

### Conversão dos Scripts para o PowerShell

Os scripts originais foram convertidos para o formato de *batch* do PowerShell, uma vez que o trabalho foi realizado em ambiente de desenvolvimento Windows.

Os novos scripts constam na pasta `scripts` do projeto, enquanto os scripts originais foram preservados na subpasta `scripts/_original_bash_scripts`.

Foram mantidos em formato bash os scripts que, na verdade, são transferidos para as instâncias de teste, guardados na subpasta `scripts/data_scripts`.

### Levantamento de Custos Operacionais

Os custos operacionais das instâncias de teste foram obtidos [no site oficial do serviço EC2](https://us-east-1.console.aws.amazon.com/ec2/home?region=us-east-1). Os dados extraídos da página de preços do EC2 foram consolidados em um DataFrame Pandas.

In [2]:
# Dados de tipos de instância baixados do site de preços oficial do EC2, com custo menos que US$0.50/hora
#
PRICES_DATA_PATH = "data/ec2-prices"
data_files = os.listdir(PRICES_DATA_PATH)
print(f"Encontrados {len(data_files)} arquivos. Iniciando extração de dados...")

df_prices = pd.DataFrame()
for csv_file in data_files:
    print(csv_file, end='... ')
    
    csv_partial_file = os.path.join(PRICES_DATA_PATH, csv_file)
    df_prices = pd.concat([df_prices, pd.read_csv(csv_partial_file)], ignore_index=True)

print(f"\nRegistradas {df_prices.shape[0]} linhas.")

Encontrados 7 arquivos. Iniciando extração de dados...
instancetypes-p1.csv... instancetypes-p2.csv... instancetypes-p3.csv... instancetypes-p4.csv... instancetypes-p5.csv... instancetypes-p6.csv... instancetypes-p7.csv... 
Registradas 339 linhas.


In [3]:
# Convertendo valor do custo e selecionando os tipos de instância candidatos.
#
PRICE_COLUMN = 'On-Demand Linux pricing'

float_extractor = lambda v: float(re.findall(r"[-+]?\d*\.\d+|\d+", str(v))[0])

df_prices['Cost'] = df_prices[PRICE_COLUMN].apply(float_extractor)
df_prices = df_prices[(df_prices['Cost'] > 0) & (df_prices['Cost'] <= 0.5)]

print(f"Filtradas {df_prices.shape[0]} linhas com custo menor que US$0.50/hora.")

Filtradas 327 linhas com custo menor que US$0.50/hora.


Com a lista de tipos de instância candidatos inicial, foram excluídas as famílias de tipos que não interessam ou não se aplicam ao problema, com base nas [descrições oficiais de cada tipo de instância](https://docs.aws.amazon.com/ec2/latest/instancetypes/instance-type-names.html), como as famílias iniciadas por `g`, voltadas para aceleração por GPU ou `h` e `d`, que utilizam armazenamento HDD, dentre outras.

Por esta regra, foram selecionados os tipos de instância `t3`, por ter sido utilizado como exemplo do problema, instâncias das famílias `c`, otimizadas para computação e algumas outras da família `r`, otimizadas para memória. Foram também removidas as famílias variantes, como, por exemplo, aquelas que especificam o processador -- `c5a`, que indica uso de AMD, ou `c6i` que especifica o uso de Intel, etc -- ou variantes que modificam o armazenamento ou rede. Do restante, selecionamos apenas a versão mais recente do tipo. Assim, restaram para análise os tipos de instância `t3`, `c5`, `m5` e `r5`.

In [4]:
INSTANCE_TYPE_COLUMN = 'Instance type'

families = sorted(
    df_prices[INSTANCE_TYPE_COLUMN].str.split('.').str[0].unique()
)

print(f"Famílias existentes: {families}")

Famílias existentes: ['a1', 'c1', 'c3', 'c4', 'c5', 'c5a', 'c5ad', 'c5d', 'c5n', 'c6a', 'c6g', 'c6gd', 'c6gn', 'c6i', 'c6id', 'c6in', 'c7a', 'c7g', 'c7gd', 'c7gn', 'c7i', 'c7i-flex', 'c8a', 'c8g', 'c8gb', 'c8gd', 'c8gn', 'c8i', 'c8i-flex', 'd3', 'g4ad', 'g5g', 'g6f', 'h1', 'i3', 'i3en', 'i4g', 'i4i', 'i7i', 'i7ie', 'i8g', 'i8ge', 'im4gn', 'inf1', 'is4gen', 'm1', 'm2', 'm3', 'm4', 'm5', 'm5a', 'm5ad', 'm5d', 'm5dn', 'm5n', 'm5zn', 'm6a', 'm6g', 'm6gd', 'm6i', 'm6id', 'm6idn', 'm6in', 'm7a', 'm7g', 'm7gd', 'm7i', 'm7i-flex', 'm8a', 'm8g', 'm8gb', 'm8gd', 'm8gn', 'm8i', 'm8i-flex', 'r3', 'r4', 'r5', 'r5a', 'r5ad', 'r5b', 'r5d', 'r5dn', 'r5n', 'r6a', 'r6g', 'r6gd', 'r6i', 'r6id', 'r6idn', 'r6in', 'r7a', 'r7g', 'r7gd', 'r7i', 'r7iz', 'r8a', 'r8g', 'r8gb', 'r8gd', 'r8gn', 'r8i', 'r8i-flex', 't1', 't2', 't3', 't3a', 't4g', 'x2gd', 'x8g', 'z1d']


In [5]:
filter = lambda x: x.startswith('t3.') or x.split('.')[0] in ['c5', 'm5', 'r5', 't3']

df_prices = df_prices[df_prices[INSTANCE_TYPE_COLUMN].apply(filter)]

print(f"Filtradas {df_prices.shape[0]} linhas com famílias de instância de interesse.")

Filtradas 14 linhas com famílias de instância de interesse.


Para cada uma das famílias candidatas, calculamos a quantidade máxima de instâncias que podem ser implantadas no orçamento de US$0.50/hora.

In [6]:
max_calculator = lambda x: int(0.5 / x)

df_prices.loc[:, 'Max Instances'] = df_prices['Cost'].apply(max_calculator)

In [7]:
df_prices [['Instance type', 'Cost', 'Max Instances']].sort_values(by='Max Instances', ascending=False)

,Instance type,Cost,Max Instances
2,t3.micro,0.0104,48
7,t3.small,0.0208,24
18,t3.medium,0.0416,12
57,t3.large,0.0832,6
61,c5.large,0.0850,5
79,m5.large,0.0960,5
111,r5.large,0.1260,3
148,t3.xlarge,0.1664,3
152,c5.xlarge,0.1700,2
176,m5.xlarge,0.1920,2


A partir da tabela de preços, foram identificadas os tipos de instância candidatos para o experimento, tendo sido selecionadas as instâncias t3.micro (tipo base, mais barato e mais leve, utilizado como exemplo na especificação do trabalho), c5.large, c5.xlarge e c5.2xlarge (tipo premium, mais caro e mais pesado). Estas escolhas visavam permitir os testes de escalabilidade horizontal e vertical, conforme definido na especificação do trabalho. 

### Experimentos

Os experimentos foram padronizados para testar cada configuração com um certo número de usuários a serem simulados com o Locust. As faixas definidas são as seguintes:

| Cenário | Quantidade de Usuários | 
| --- | --- |
| Uso Mínimo | 100 | 
| Uso Baixo | 250 |
| Uso Médio | 500 |
| Uso Alto | 1000 |
| Uso Muito Alto | 2000 |

Salinta-se que nem todas as faixas foram testadas com todos os tipos de instâncias. No caso, o uso médio é adotado como referencial para os experimentos de escalonamento. 

Além disso, os experimentos foram realizados com um tempo de duração de 3 minutos.

#### Fase 1  Escalabilidade Horizontal

Para a escalabilidade horizontal, apenas podemos considerar os tipos de instância que permitem se utilizar mais de uma instância. Neste caso, em vez de testes exaustivos aumentando a quantidade de instâncias de uma em uma, optamos por fazer uso de três amostras para cada tipo de instância, conforme o desempenho obtido.

Iniciamos executando a configuração base (*baseline*), que consiste em uma instância t3.micro com 100 usuários.

In [8]:
def evaluate_results(df_locust: pd.DataFrame):
    """
    Avalia os resultados de um teste de carga com locust.
    """
    df_locust['Error Rate'] = df_locust['Failure Count'] / df_locust['Request Count']
    df_locust['Error Rate Pass'] = df_locust['Error Rate'] < 0.01
    df_locust['P95 Pass'] = df_locust['95%'] < 10000
    df_locust['Total Pass'] = df_locust['Error Rate Pass'] & df_locust['P95 Pass']

    return df_locust[['Type', 'Name', 'Request Count', 'Failure Count', 'Error Rate', 'Error Rate Pass', '95%', 'P95 Pass', 'Total Pass']]


def evaluate_results_from_csv(file_path: str):
    df_locust = pd.read_csv(file_path)
    return evaluate_results(df_locust)

In [9]:
df_baseline = pd.read_csv('results/ProjectExampleBaseline_stats.csv')

evaluate_results(df_baseline)

,Type,Name,Request Count,Failure Count,Error Rate,Error Rate Pass,95%,P95 Pass,Total Pass
0,GET,/,610,598,0.980328,False,8400,True,False
1,GET,/post-detail,1995,1970,0.987469,False,8500,True,False
2,NaN,Aggregated,2605,2568,0.985797,False,8500,True,False


A configuração *baseline*, embora atenda ao requisito mínimo de latência P95, possui taxa de erro de quase 100%, não satisfazendo os requisitos de desempenho. Resolvemos então testar a escalabilidade horizontal da `t3.micro` elevando o número de instâncias para 12 (um quarto do máximo de instâncias disponíveis) e verificando sua capacidade de atender a 100 usuários.

In [10]:
evaluate_results_from_csv('results/t3-micro_100u_12i_stats.csv')

,Type,Name,Request Count,Failure Count,Error Rate,Error Rate Pass,95%,P95 Pass,Total Pass
0,GET,/,1241,85,0.068493,False,3000,True,False
1,GET,/post-detail,4243,301,0.070940,False,3000,True,False
2,NaN,Aggregated,5484,386,0.070387,False,3000,True,False


A taxa de erros com 12 instâncias reduziu-se substancialmnente, assim como a latência P95. Entretanto, a taxa de erros ainda não atingiu o parâmetro exigido de menos de 1%. Vamos aumentar para 24 instâncias, metade do limite máximo de 48 instâncias. 

In [11]:
evaluate_results_from_csv('results/t3-micro_100u_24i_stats.csv')

,Type,Name,Request Count,Failure Count,Error Rate,Error Rate Pass,95%,P95 Pass,Total Pass
0,GET,/,1658,0,0.0,True,650,True,True
1,GET,/post-detail,5766,0,0.0,True,620,True,True
2,NaN,Aggregated,7424,0,0.0,True,620,True,True


Agora sim. Com 24 instâncias t3.micro, conseguimos atender aos requisitos exigidos pelo experimento. Neste caso, o custo operacional é:

In [12]:
def cost_of(type: str, instances: int) -> str:
    """
    Calcula o custo horário de uma configuração de instâncias EC2.
    """
    ucost = df_prices[df_prices[INSTANCE_TYPE_COLUMN] == type]['Cost'].values[0]
    cost = ucost * instances
    return f"US$ {cost:.2f}"


In [13]:
cost_of('t3.micro', 24)

'US$ 0.25'

Vamos verificar como esta configuração se comporta com 250 usuários.

In [14]:
evaluate_results_from_csv('results/t3-micro_250u_24i_stats.csv')

,Type,Name,Request Count,Failure Count,Error Rate,Error Rate Pass,95%,P95 Pass,Total Pass
0,GET,/,2646,1125,0.425170,False,4100,True,False
1,GET,/post-detail,9004,3800,0.422035,False,4100,True,False
2,NaN,Aggregated,11650,4925,0.422747,False,4100,True,False


Com 250 usuários, os critérios exigidos não são mais atendidos pela configuração. Vamos extrapolar o uso da `t3.micro` testando-a com 48 instâncias (o máximo possível) para 1000 usuários.

In [15]:
evaluate_results_from_csv('results/t3-micro_1000u_48i_stats.csv')

,Type,Name,Request Count,Failure Count,Error Rate,Error Rate Pass,95%,P95 Pass,Total Pass
0,GET,/,9804,7225,0.736944,False,11000,False,False
1,GET,/post-detail,32891,24044,0.731021,False,11000,False,False
2,NaN,Aggregated,42695,31269,0.732381,False,11000,False,False


Novamente, os resultados são muito ruins. Nossos resultados apontam para que o uso da `t3.micro` não é viável para este caso de uso. Na verdade, instâncias das famílias `t` são, em geral, utilizadas como máquinas de teste ou desenvolvimento, e não de produção. De fato, constatamos que, para atender a demanda de meros 100 usuários, foram necessárias cerca de 24 instâncias. Ainda que um número menor de instâncias pudesse atender a demanda, ainda assim, trata-se de um tipo de instância muito básico, dificilmente adequado para cenários de produção.

De toda forma, vamos seguir testando a família `t3` identificando qual tipo seria capaz de atender 500 usuários dentro dos critérios de desempenho exigidos.

In [16]:
df_prices[df_prices['Instance type'].str.contains('t3')][['Instance type', 'Cost', 'Max Instances']].sort_values(by='Max Instances', ascending=True)

,Instance type,Cost,Max Instances
253,t3.2xlarge,0.3328,1
148,t3.xlarge,0.1664,3
57,t3.large,0.0832,6
18,t3.medium,0.0416,12
7,t3.small,0.0208,24
2,t3.micro,0.0104,48


Iniciamos os testes utilizando metade da quantidade de instâncias possíveis.

In [17]:
evaluate_results_from_csv('results/t3-small_500u_12i_stats.csv')

,Type,Name,Request Count,Failure Count,Error Rate,Error Rate Pass,95%,P95 Pass,Total Pass
0,GET,/,2408,862,0.357973,False,38000,False,False
1,GET,/post-detail,7892,2826,0.358084,False,38000,False,False
2,NaN,Aggregated,10300,3688,0.358058,False,38000,False,False


In [18]:
evaluate_results_from_csv('results/t3-medium_500u_6i_stats.csv')

,Type,Name,Request Count,Failure Count,Error Rate,Error Rate Pass,95%,P95 Pass,Total Pass
0,GET,/,1248,436,0.349359,False,60000,False,False
1,GET,/post-detail,4192,1427,0.340410,False,60000,False,False
2,NaN,Aggregated,5440,1863,0.342463,False,60000,False,False


In [19]:
evaluate_results_from_csv('results/t3-large_500u_3i_stats.csv')

,Type,Name,Request Count,Failure Count,Error Rate,Error Rate Pass,95%,P95 Pass,Total Pass
0,GET,/,1488,936,0.629032,False,60000,False,False
1,GET,/post-detail,5017,3112,0.620291,False,60000,False,False
2,NaN,Aggregated,6505,4048,0.622291,False,60000,False,False


In [20]:
evaluate_results_from_csv('results/t3-xlarge_500u_1i_stats.csv')

,Type,Name,Request Count,Failure Count,Error Rate,Error Rate Pass,95%,P95 Pass,Total Pass
0,GET,/,1807,1429,0.790814,False,10000,False,False
1,GET,/post-detail,6107,4809,0.787457,False,10000,False,False
2,NaN,Aggregated,7914,6238,0.788223,False,10000,False,False


Algumas observações sobre os resultados:

- Nenhuma das configurações conseguiu atingir os objetivos de performance estabelecidos.
- Embora os tipos de instância sejam progressivamente mais poderosos, a performance não melhora proporcionalmente com a escalabilidade. De fato, metade da quantidade máxima de instâncias `t3-small` têm latência P95 menor que as metades das quantidades máximas de instâncias de `t3-medium` e `t3-large`. No caso desta última, a taxa de erros é quase o dobro que a dos tipos anteriores.
- Embora se aproxime do limite estabelecido para P95, o tipo `t3-xlarge` não atinge a taxa de erros desejada, chegando a quase 80% de erro.

Vamos avaliar novamente estas configurações com o máximo de instâncias possível para cada uma. Aproveitamos para fazer o teste com a `t3.2xlarge`, da qual somente temos disponibilidade de uma instância, respeitando nosso limite de custo.

In [21]:
evaluate_results_from_csv('results/t3-small_500u_24i_stats.csv')

,Type,Name,Request Count,Failure Count,Error Rate,Error Rate Pass,95%,P95 Pass,Total Pass
0,GET,/,2724,1116,0.409692,False,22000,False,False
1,GET,/post-detail,8868,3807,0.429296,False,21000,False,False
2,NaN,Aggregated,11592,4923,0.424689,False,21000,False,False


In [22]:
evaluate_results_from_csv('results/t3-medium_500u_12i_stats.csv')

,Type,Name,Request Count,Failure Count,Error Rate,Error Rate Pass,95%,P95 Pass,Total Pass
0,GET,/,2303,858,0.372558,False,36000,False,False
1,GET,/post-detail,8010,3088,0.385518,False,35000,False,False
2,NaN,Aggregated,10313,3946,0.382624,False,36000,False,False


In [23]:
evaluate_results_from_csv('results/t3-large_500u_6i_stats.csv')

,Type,Name,Request Count,Failure Count,Error Rate,Error Rate Pass,95%,P95 Pass,Total Pass
0,GET,/,1103,246,0.223028,False,60000,False,False
1,GET,/post-detail,3725,805,0.216107,False,60000,False,False
2,NaN,Aggregated,4828,1051,0.217688,False,60000,False,False


In [24]:
evaluate_results_from_csv('results/t3-xlarge_500u_3i_stats.csv')

,Type,Name,Request Count,Failure Count,Error Rate,Error Rate Pass,95%,P95 Pass,Total Pass
0,GET,/,1683,629,0.373737,False,60000,False,False
1,GET,/post-detail,5651,2060,0.364537,False,60000,False,False
2,NaN,Aggregated,7334,2689,0.366648,False,60000,False,False


In [25]:
evaluate_results_from_csv('results/t3-2xlarge_500u_1i_stats.csv')

,Type,Name,Request Count,Failure Count,Error Rate,Error Rate Pass,95%,P95 Pass,Total Pass
0,GET,/,2056,1327,0.645428,False,10000,False,False
1,GET,/post-detail,6824,4270,0.625733,False,10000,False,False
2,NaN,Aggregated,8880,5597,0.630293,False,10000,False,False


As observações que se pode depreender dos resultados obtidos para 500 usuários são:

- No caso da `t3.small` e `t3.medium`, houve melhoria de P95 ao custo de aumento nas taxas de erro. 
- No caso da `t3.large`, houve redução da taxa de erro sem prejuízo da latência P95.
- No caso da `t3.xlarge`, houve melhoria da taxa de erro com sério comprometimento de P95.
- No caso da `t3.2xlarge`, não há comparação anterior, mas as taxas de erro se apresentam como as maiores dentre o grupo de tipos de instância.
- **Nenhuma das configurações testadas nesta seção atende aos requisitos de latência e taxa de erro.**

**Conclusão:**
- As famílias `t` são, definitivamente, mais apropriadas para uso em fase de desenvolvimento e testes, e não para ambientes de produção.

Sigamos para os testes de escalabilidade horizontal de instâncias das famílias `m5`, a saber:

In [28]:
df_prices[df_prices[INSTANCE_TYPE_COLUMN].str.startswith('m5')][[INSTANCE_TYPE_COLUMN, 'Cost', 'Max Instances']]

,Instance type,Cost,Max Instances
79,m5.large,0.096,5
176,m5.xlarge,0.192,2
285,m5.2xlarge,0.384,1


Os resultados com o máximo de instâncias possível para 500 usuários são apresentados nas tabelas adiante.

In [29]:
evaluate_results_from_csv('results/m5-large_500u_5i_stats.csv')

,Type,Name,Request Count,Failure Count,Error Rate,Error Rate Pass,95%,P95 Pass,Total Pass
0,GET,/,1229,428,0.348251,False,60000,False,False
1,GET,/post-detail,4277,1499,0.350479,False,60000,False,False
2,NaN,Aggregated,5506,1927,0.349982,False,60000,False,False


In [32]:
evaluate_results_from_csv('results/m5-xlarge_500u_2i_stats.csv')

,Type,Name,Request Count,Failure Count,Error Rate,Error Rate Pass,95%,P95 Pass,Total Pass
0,GET,/,2107,1175,0.557665,False,10000,False,False
1,GET,/post-detail,6851,3731,0.544592,False,10000,False,False
2,NaN,Aggregated,8958,4906,0.547667,False,10000,False,False


In [33]:
evaluate_results_from_csv('results/m5-2xlarge_500u_1i_stats.csv')

,Type,Name,Request Count,Failure Count,Error Rate,Error Rate Pass,95%,P95 Pass,Total Pass
0,GET,/,2101,1317,0.626844,False,10000,False,False
1,GET,/post-detail,6807,4118,0.604965,False,10000,False,False
2,NaN,Aggregated,8908,5435,0.610126,False,10000,False,False


Conclusões acerca dos testes com as instâncias da família `m5`:

- Novamente, nenhuma configuração foi capaz de suportar os 500 usuários dentro dos parâmetros exigidos.
- A partir da configuração `m5-xlarge`, a taxa de P95 aproximou-se do valor desejado.
- A confoguração `m5-2xlarge` apresentou as piores taxas de erro dentre os três tipos testados.

Considerando que a família `m5` é a família "de produção" correspondente a `t3`, devemos considerar a hipótese de que 500 usuários é muita coisa para o nosso orçamento limitado. Neste caso, devemos considerar reduzir o número de usuários, passando a testar com 250 usuários. Apesar disso, não vale a pena reconsiderar o uso da família `t3` pois já determinamos que não se adequa ao uso em produção.

Vamos ver como a família `m5` se comporta com 250 usuários.


In [34]:
evaluate_results_from_csv('results/m5-large_250u_5i_stats.csv')

,Type,Name,Request Count,Failure Count,Error Rate,Error Rate Pass,95%,P95 Pass,Total Pass
0,GET,/,860,60,0.069767,False,60000,False,False
1,GET,/post-detail,2734,186,0.068032,False,60000,False,False
2,NaN,Aggregated,3594,246,0.068447,False,60000,False,False


In [35]:
evaluate_results_from_csv('results/m5-xlarge_250u_2i_stats.csv')

,Type,Name,Request Count,Failure Count,Error Rate,Error Rate Pass,95%,P95 Pass,Total Pass
0,GET,/,877,186,0.212087,False,60000,False,False
1,GET,/post-detail,2874,673,0.234168,False,60000,False,False
2,NaN,Aggregated,3751,859,0.229006,False,60000,False,False


In [36]:
evaluate_results_from_csv('results/m5-2xlarge_250u_1i_stats.csv')

,Type,Name,Request Count,Failure Count,Error Rate,Error Rate Pass,95%,P95 Pass,Total Pass
0,GET,/,1176,387,0.329082,False,10000,False,False
1,GET,/post-detail,3929,1203,0.306185,False,10000,False,False
2,NaN,Aggregated,5105,1590,0.311459,False,10000,False,False


Conclusões para esta experiência de 250 usuários com a família `m5`:

- Embora as taxas de erro tenham sido drasticamente reduzidas, a métrica ainda não atinge o objetivo de 1%.
- A latência de P95 permanece bastante alta, inviabilizando o uso da família `m5` com esta configuração de 250 usuários.

Desta forma, vamos, novamente, reduzir o quantitativo de usuários de referência, adotando o mínimo de 100 usuários.

Testando novamente com a família `m5` e 100 usuários:

In [37]:
evaluate_results_from_csv('results/m5-large_100u_5i_stats.csv')

,Type,Name,Request Count,Failure Count,Error Rate,Error Rate Pass,95%,P95 Pass,Total Pass
0,GET,/,515,15,0.029126,False,23000,False,False
1,GET,/post-detail,1713,64,0.037361,False,52000,False,False
2,NaN,Aggregated,2228,79,0.035458,False,51000,False,False


In [39]:
evaluate_results_from_csv('results/m5-xlarge_100u_2i_stats.csv')

,Type,Name,Request Count,Failure Count,Error Rate,Error Rate Pass,95%,P95 Pass,Total Pass
0,GET,/,706,10,0.014164,False,21000,False,False
1,GET,/post-detail,2367,33,0.013942,False,15000,False,False
2,NaN,Aggregated,3073,43,0.013993,False,15000,False,False


In [40]:
evaluate_results_from_csv('results/m5-2xlarge_100u_1i_stats.csv')

,Type,Name,Request Count,Failure Count,Error Rate,Error Rate Pass,95%,P95 Pass,Total Pass
0,GET,/,919,7,0.007617,True,15000,False,False
1,GET,/post-detail,2718,24,0.008830,True,2800,True,True
2,NaN,Aggregated,3637,31,0.008524,True,2900,True,True
